In [8]:
# ==============================================================================
# CELL 1: Setup Environment & Download Model Weights
# ==============================================================================
from google.colab import drive, userdata
import os

# 1. Mount Drive and Set Cache
CACHE_DIR = "/content/drive/MyDrive/huggingface_cache"
os.makedirs(CACHE_DIR, exist_ok=True)
os.environ["HF_HOME"] = CACHE_DIR

# 2. Install minimal requirements (Keep pre-installed PyTorch intact)
!pip install -q -U transformers accelerate huggingface_hub pyngrok flask flask-cors pillow requests

# 3. Authenticate & Download MedGemma 1.5
from huggingface_hub import login, snapshot_download
from pyngrok import ngrok

HF_TOKEN = userdata.get('HF_TOKEN')
NGROK_TOKEN = userdata.get('NGROK_AUTH_TOKEN')

login(token=HF_TOKEN)
ngrok.set_auth_token(NGROK_TOKEN)

MODEL_ID = "google/medgemma-1.5-4b-it"
SAVE_PATH = f"{CACHE_DIR}/{MODEL_ID.split('/')[-1]}"

print(f"Downloading {MODEL_ID} to Google Drive...")
snapshot_download(repo_id=MODEL_ID, local_dir=SAVE_PATH, token=HF_TOKEN)
print(f"✅ Model cached at: {SAVE_PATH}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.8/795.8 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 96.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

✅ Model cached at: /content/drive/MyDrive/huggingface_cache/medgemma-1.5-4b-it


In [ ]:
import os, io, gc, base64
import torch
from PIL import Image
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
from google.colab import userdata
from transformers import AutoProcessor, AutoModelForImageTextToText

# 1. Environment Setup & Authentication
HF_TOKEN = userdata.get('HF_TOKEN')
NGROK_TOKEN = userdata.get('NGROK_AUTH_TOKEN')

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HF_HOME"] = "/content/drive/MyDrive/huggingface_cache"

from huggingface_hub import login
login(token=HF_TOKEN)
ngrok.set_auth_token(NGROK_TOKEN)

# 2. Model Loading Helper
MODEL_PATH = "/content/drive/MyDrive/huggingface_cache/medgemma-1.5-4b-it"

processor = None
model = None

def load_medgemma():
    global processor, model
    if model is None:
        print("🚀 Loading MedGemma 1.5 weights onto GPU...")
        processor = AutoProcessor.from_pretrained(MODEL_PATH)
        model = AutoModelForImageTextToText.from_pretrained(
            MODEL_PATH,
            torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
            device_map="auto"
        )
        print(f"✅ Loaded successfully on {model.device}!")

load_medgemma()

# 3. Flask Server Initialization
app = Flask(__name__)
CORS(app)

@app.after_request
def add_ngrok_headers(response):
    response.headers["ngrok-skip-browser-warning"] = "true"
    return response

@app.route("/health", methods=["GET"])
def health():
    return jsonify({
        "status": "ok",
        "device": str(model.device) if model else "unloaded",
        "model_loaded": model is not None
    })

@app.route("/gpu-status", methods=["GET"])
def gpu_status():
    """Returns exact VRAM usage in MB."""
    if not torch.cuda.is_available():
        return jsonify({"cuda_available": False})

    allocated = torch.cuda.memory_allocated() / (1024 ** 2)
    reserved = torch.cuda.memory_reserved() / (1024 ** 2)
    max_allocated = torch.cuda.max_memory_allocated() / (1024 ** 2)

    return jsonify({
        "cuda_available": True,
        "allocated_vram_mb": round(allocated, 2),
        "reserved_vram_mb": round(reserved, 2),
        "max_allocated_vram_mb": round(max_allocated, 2)
    })

@app.route("/generate", methods=["POST"])
def generate():
    global processor, model
    if model is None:
        load_medgemma()

    inputs = None
    generation = None
    output_tokens = None

    try:
        data = request.get_json(force=True) or {}
        prompt = data.get("prompt") or data.get("inputs") or data.get("text") or ""
        max_tokens = int(data.get("max_tokens", 512))
        temperature = float(data.get("temperature", 0.1))
        image_b64 = data.get("image_base64") or data.get("image")

        if not prompt and not image_b64:
            return jsonify({"error": "Missing 'prompt' or 'image' field"}), 400

        # Construct multimodal input message
        content = []
        if image_b64:
            if "," in image_b64:
                image_b64 = image_b64.split(",")[1]
            img_bytes = base64.b64decode(image_b64)
            img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
            content.append({"type": "image", "image": img})

        content.append({"type": "text", "text": prompt if prompt else "Analyze this medical document or image."})
        messages = [{"role": "user", "content": content}]

        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt"
        )
        inputs = {k: (v.to(model.device) if isinstance(v, torch.Tensor) else v) for k, v in inputs.items()}
        input_len = inputs["input_ids"].shape[-1]

        gen_kwargs = {
            "max_new_tokens": max_tokens,
            "do_sample": temperature > 0.0
        }
        if temperature > 0.0:
            gen_kwargs["temperature"] = temperature
            gen_kwargs["top_p"] = 0.9

        with torch.inference_mode():
            generation = model.generate(**inputs, **gen_kwargs)
            output_tokens = generation[0][input_len:]

        response_text = processor.decode(output_tokens, skip_special_tokens=True)

        return jsonify({
            "response": response_text.strip(),
            "tokens_generated": len(output_tokens),
            "status": "success"
        })

    except Exception as e:
        return jsonify({"error": str(e)}), 500

    finally:
        # POST-RESPONSE GPU VRAM CLEANUP
        del inputs, generation, output_tokens
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

@app.route("/unload", methods=["POST"])
def unload():
    """Evicts MedGemma from VRAM completely."""
    global processor, model
    model = None
    processor = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("🧹 VRAM cleared. Model unloaded.")
    return jsonify({"status": "success", "message": "Model evicted from GPU memory."})

# 4. Open Tunnel and Start App
ngrok.kill()
tunnel = ngrok.connect(5000, "http")
print(f"\n==================================================")
print(f"🔗 COLAB_API_URL = {tunnel.public_url}")
print(f"==================================================\n")

app.run(host="0.0.0.0", port=5000)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


🚀 Loading MedGemma 1.5 weights onto GPU...


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

✅ Loaded successfully on cuda:0!

🔗 COLAB_API_URL = https://unilludedly-pipier-paola.ngrok-free.dev

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
